### Caso não tenha as libs instaladas no Kernel

In [1]:
%pip install plotly pandas
%pip install --upgrade nbformat

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### Import das libs

In [6]:
import pandas as pd
import plotly.graph_objects as go
import glob
import os

### Análise dos dados

In [8]:
lista_dfs = []
caminhos_ficheiros = glob.glob('logs/*.csv')

for caminho in caminhos_ficheiros:
    df_temp = pd.read_csv(caminho)
    
    df_temp['altitude'] = -df_temp['z']
    
    lista_dfs.append(df_temp)

#### Análise da trajetória

In [9]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    
    # Extraindo o timestamp do nome do arquivo
    nome_arquivo = os.path.basename(caminho) 
    timestamp = nome_arquivo.replace('voo_teste_', '').replace('.csv', '')
    
    fig_3d = go.Figure()

    # Adiciona a linha da Trajetória Real
    fig_3d.add_trace(go.Scatter3d(
        x=df['x'], y=df['y'], z=df['altitude'],
        mode='lines',
        line=dict(color='royalblue', width=4),
        name='Trajetória Real (Odometria)'
    ))

    # Adiciona o Ponto de Partida
    fig_3d.add_trace(go.Scatter3d(
        x=[df['x'].iloc[0]], y=[df['y'].iloc[0]], z=[df['altitude'].iloc[0]],
        mode='markers',
        marker=dict(color='green', size=6),
        name='Ponto de Partida'
    ))

    # Adiciona o Ponto Final
    fig_3d.add_trace(go.Scatter3d(
        x=[df['x'].iloc[-1]], y=[df['y'].iloc[-1]], z=[df['altitude'].iloc[-1]],
        mode='markers',
        marker=dict(color='red', size=6, symbol='x'),
        name='Ponto Final da Run'
    ))

    fig_3d.update_layout(
        title=f'Análise de Trajetória 3D do VANT - Run: {timestamp}',
        scene=dict(
            xaxis_title='Posição X (Metros)',
            yaxis_title='Posição Y (Metros)',
            zaxis_title='Posição Z / Altitude (Metros)',
            camera=dict(eye=dict(x=1.5, y=1.5, z=0.5)) 
        ),
        legend=dict(x=0, y=1),
        margin=dict(l=0, r=0, b=0, t=40) 
    )

    fig_3d.show()

#### Análise do Pitch and Roll

In [11]:
for caminho, df in zip(caminhos_ficheiros, lista_dfs):
    
    # Extraindo o timestamp do nome do arquivo
    nome_arquivo = os.path.basename(caminho) 
    timestamp = nome_arquivo.replace('voo_teste_', '').replace('.csv', '')
    
    fig_2d = go.Figure()

    # Adiciona a linha de Roll
    fig_2d.add_trace(go.Scatter(
        x=df['timestamp'], y=df['roll_speed'],
        mode='lines',
        name='Roll',
        opacity=0.7
    ))

    # Adiciona a linha de Pitch
    fig_2d.add_trace(go.Scatter(
        x=df['timestamp'], y=df['pitch_speed'],
        mode='lines',
        name='Pitch',
        opacity=0.7
    ))

    fig_2d.update_layout(
        title=f'Esforço de Controle: Velocidades Angulares - Run: {timestamp}',
        xaxis_title='Tempo de Voo (Segundos)',
        yaxis_title='Velocidade Angular (rad/s)',
        template='plotly_white',
        hovermode='x unified' # Cria uma linha vertical interativa ao passar o mouse
    )

    fig_2d.show()